In [ ]:
from pathlib import Path

import networkx as nx
import osmnx as ox
import pandas as pd
from scipy import stats

In [ ]:
df_ind = pd.read_csv("indicators-all.csv")
df_ind.shape

In [ ]:
df_mst = pd.read_csv("mst-results.csv")
df_mst["uc_id"] = df_mst["id"].str.split("-", expand=True)[1].astype(int)
df_mst = df_mst.drop(columns=["id"])
df_mst.shape

In [ ]:
df = df_ind.merge(df_mst, on="uc_id", how="inner")
df["intersect_density"] = df["intersect_count"] / df["built_up_area_m2"]
df.shape

In [ ]:
df = df[df["node_count"] > 100]
df.shape

In [ ]:
df_mst

In [ ]:
# whole graph vs connected component makes little difference
cols = [
    "edge_count_ratio",
    "length_ratio",
    "edge_count_ratio_cc",
    "length_ratio_cc",
    "cc_length_ratio",
    "treewidth",
    "treewidth_cc",
]
df[cols].describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(5, 5))
bins = 40
alpha = 0.6
ax = df["length_ratio"].hist(bins=bins, alpha=alpha, ax=ax, color="#003399", label="Length Ratio")
ax = df["edge_count_ratio"].hist(
    bins=bins, alpha=alpha, ax=ax, color="#cc0000", label="Edge Count Ratio"
)
ax.set_xlim(0, 1)
_ = plt.legend()
fig.savefig("histograms-lr-ecr.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
bins = 40
alpha = 0.7
ax = df["circuit_rank_ratio"].hist(
    bins=bins, alpha=alpha, ax=ax, color="#cc0000", label="Circuit Rank Ratio"
)
ax.set_xlim(0, 1)
_ = plt.legend()
fig.savefig("histogram-cr.png", dpi=300, bbox_inches="tight")

In [ ]:
cols = ["edge_count_ratio", "length_ratio", "treewidth"]
df.groupby("world_region")[cols].mean().sort_values("length_ratio").round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = 30
alpha = 0.7
for label, group in df.groupby("world_region"):
    ax = group["length_ratio"].hist(bins=bins, alpha=alpha, ax=ax, label=label, zorder=-len(group))
ax.set_xlim(0, 1)
ax.set_xlabel("Length Ratio")
_ = plt.legend()

In [ ]:
# edge_count_ratio + circuit_rank_ratio should equal 1, because...
# circuit_rank_ratio = (edge_count - node_count + component_count) / edge_count
# edge_count_ratio = edge_count_mst / edge_count_original = 1 - circuit_rank_ratio
(df["edge_count_ratio"] + df["circuit_rank_ratio"]).describe().round(4)

In [ ]:
str(sorted(df.columns))

In [ ]:
# calculate correlations
cols = [
    "node_count",
    "length_total",
    "orientation_entropy",
    "bc_gini",
    "area_km2",
    "k_avg",
    "elev_median",
    "straightness",
]
corrs = []
for x_col in ["edge_count_ratio", "length_ratio"]:
    for y_col in cols:
        r, p = stats.pearsonr(df[x_col], df[y_col])
        corrs.append((x_col, y_col, r, p))

# show results
cols = ["x_col", "y_col", "r", "p"]
df_corrs = pd.DataFrame(corrs, columns=cols)  # .set_index(["x_col", "y_col"])
df_corrs.sort_values(by="r", key=abs, ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), facecolor="w")
x_cols = ["k_avg", "bc_gini", "orientation_entropy"]
x_labels = ["Avg Degree", "Gini (BC)", "Orientation Entropy"]
y_cols = ["length_ratio", "edge_count_ratio"]
y_labels = ["Length Ratio", "Edge Count Ratio"]
xlims = [(1.9, 3.7), (0.38, 0.95), (2, 3.58)]
colors = ["#003399", "#cc0000"]
for x_col, ax, xlim, x_label in zip(x_cols, axes, xlims, x_labels, strict=False):
    title = ""
    for y_col, color, y_label in zip(y_cols, colors, y_labels, strict=False):
        ax.scatter(df[x_col], df[y_col], s=1, alpha=0.05, color=color)

        m, b, r, p, se = stats.linregress(df[x_col], df[y_col])
        ax.axline((0, b), slope=m, color=color, linestyle="--", linewidth=1, label=y_label)

        ax.set_xlabel(x_label)

        ax.set_xlim(xlim)
        ax.set_ylim(0.3, 1)

        title = title + f"{y_col}={m:.2f}x+{b:.2f}\n"

    # ax.set_title(title.strip("\n"))

_ = ax.legend()
fig.savefig("scatter.png", dpi=300, bbox_inches="tight")

In [ ]:
count = 3  # how many of each?
min_edges = 600
max_edges = 900
cols = [
    "country",
    "country_iso",
    "core_city",
    "uc_id",
    "node_count",
    "edge_count_ratio_cc",
    "length_ratio_cc",
]
mask = (df["edge_count_orig_cc"] > min_edges) & (df["edge_count_orig_cc"] < max_edges)
df_subset = df[mask][cols].sort_values("length_ratio_cc")
uc_ids = df_subset["uc_id"].head(count).tolist() + df_subset["uc_id"].tail(count).tolist()
# uc_ids = [1997, 11259]#[7493, 11151]

In [ ]:
df_subset

In [ ]:
bgcolor = "#262626"
fig, axes = plt.subplots(2, count, figsize=(count * 4, 8), facecolor=bgcolor)
for uc_id, ax in zip(uc_ids, axes.flat, strict=False):
    row = df[df["uc_id"] == uc_id].iloc[0]
    root = Path("/data/snm/models/graphml")
    fp = (
        root
        / f"{row['country']}-{row['country_iso']}"
        / f"{row['core_city']}-{row['uc_id']}.graphml"
    )
    Gu = ox.convert.to_undirected(ox.truncate.largest_component(ox.io.load_graphml(fp)))
    mst = nx.minimum_spanning_tree(Gu, weight="length")
    print(row["length_ratio"])
    ax.set_facecolor(bgcolor)
    fig, ax = ox.plot.plot_graph(
        Gu, node_size=0, ax=ax, edge_color="r", bgcolor=bgcolor, show=False, close=False
    )
    fig, ax = ox.plot.plot_graph(
        mst,
        node_size=0,
        ax=ax,
        edge_linewidth=1.5,
        bgcolor=bgcolor,
        edge_color="w",
        show=False,
        close=False,
    )
fig.savefig("networks.png", dpi=200)

In [ ]:
# Treewidth
# https://en.wikipedia.org/wiki/Treewidth
# https://networkx.org/documentation/stable/reference/algorithms/approximation.html#module-networkx.algorithms.approximation.treewidth
# https://math.stackexchange.com/questions/3335762/measures-for-how-tree-like-a-graph-is?rq=1
#
# Circuit Rank
# circuit_rank = edge_count - node_count + component_count
# circuit_rank / edge_count = 1 - edge_count_ratio
# https://en.wikipedia.org/wiki/Circuit_rank
# https://math.stackexchange.com/questions/3335762/measures-for-how-tree-like-a-graph-is?rq=1
# AKA first betti number? https://math.stackexchange.com/questions/2102889/measuring-tree-ness-and-other-graph-properties

# levinson: treeness: https://onlinelibrary-wiley-com.libproxy2.usc.edu/doi/full/10.1111/j.1538-4632.2007.00707.x
# and derived from the above: https://www-sciencedirect-com.libproxy2.usc.edu/science/article/pii/S0965856417302549

# yikes! david used similar language to my idea 10 years ago!
# https://www.transportist.net/p/sometimes-a-city-is-a-tree